# ARC Prize 2026 — v12: NVARC Consensus Selection + Tuned D8 Geometric CPU Fallback (CPU Safe)

**Enhancements in v12:**
1. **Tuned CPU Fallback Protection (`cpu_fallback.py`)**: Fully exports `fill_submission_with_fallbacks()` intercepting dummy `[[0]]` grids on CPU runners. Implements D8 dihedral symmetries (rot90, rot180, rot270, reflections), color mapping compositions, integer scaling (2x, 3x), and bounding box cropping.
2. **Consensus-First Pass@2 Selection (`score_consensus_v7`)**: Restores winning strategy where both `attempt_1` and `attempt_2` are drawn strictly from augmentation consensus (`score_kgmon_calibrated`).
3. **Safe Beam Threshold (`max_score = -log(0.20)`) & `n=16` Augmentations**: Tight exploration bound and proven schedule within 1200s timeout.
4. **100% Prize-Eligible**: Fully reproducible under CC BY 4.0.


In [ ]:
# Keep the original global 10-minute submission/write buffer.
import time
global_end_time = time.time() + 12 * 3600 - 600

In [ ]:
# Preserve the baseline environment workaround.
!pip uninstall -y tensorflow

In [ ]:
%%writefile arc_loader.py
import json
import numpy as np
from transformers import AutoTokenizer


def convert_grid_to_string(grid) -> str:
    text = ""
    for row in grid:
        for cell in row:
            text += str(int(cell))
        text += "\n"
    return text.strip()

def is_valid_solution(guess):
    return isinstance(guess, np.ndarray) and guess.ndim == 2 and all(0 < x <= 30 for x in guess.shape)

def shuffled(data_list):
    return np.random.permutation(data_list).tolist()

def permute_mod(a, descriptor, invert=False):
    permutation = [int(i) for i in descriptor if str(i).isdigit()]
    assert sorted(permutation)==list(range(10))
    a = np.asarray(a)
    if a.ndim==3:
        if not invert: permutation = np.argsort(permutation)
        a = a[..., permutation]
    else:
        assert a.ndim==2
        if invert: permutation = np.argsort(permutation)
        a = np.asarray(permutation)[a]
    return a

def permute_rnd_all_(query):
    permutation = np.random.permutation(10).tolist()
    return 'permute' + ''.join(map(str, permutation))


class QwenFormatter:

    def __init__(self, tokenizer: AutoTokenizer):
        self.tokenizer = tokenizer

    def fmt_query(self, query) -> str:
        grid_input = convert_grid_to_string(query[0]["input"])
        return "<|im_start|>user\n" + grid_input + "<|im_end|><|im_start|>assistant\n"

    def fmt_reply(self, reply) -> str:
        return convert_grid_to_string(reply[0]) + "<|im_end|>"

    def fmt_train(self, train, last_is_challenge=False) -> str:
        if last_is_challenge:
            test = train[-1]
            train = train[:-1]
        else:
            test = None
        text = ""
        for x in train:
            grid_input = convert_grid_to_string(x["input"])
            grid_output = convert_grid_to_string(x["output"])
            text += f"<|im_start|>user\n{grid_input}<|im_end|><|im_start|>assistant\n{grid_output}<|im_end|>"
        if test is not None:
            text += self.fmt_query([test]) + self.fmt_reply([test["output"]])
        return text

    def max_new_tokens(self):
        max_sized_reply = np.zeros([30, 30], dtype=int)
        tokens = self.tokenizer.encode(self.fmt_reply([max_sized_reply]))
        return len(tokens) + 1

    def convert_tokens_to_array(self, tokens, limit_rows=30):
        if len(tokens) < 2:
            return None
        text = self.tokenizer.decode(tokens[:-1])
        try:
            lines = text.strip().split("\n")
            by_rows = [row for row in [[int(x) for x in line if x.isdigit()] for line in lines] if len(row)]
            if len(by_rows) > limit_rows:
                by_rows = by_rows[:limit_rows]
            array = np.array(by_rows, dtype=int)
            if is_valid_solution(array):
                return array
        except:
            pass
        return None


class ArcDataset:

    @staticmethod
    def forward_mod(a, key, use_perm=True):
        if a is None: return a
        for op in key.split('.')[1:]:
            if   op=='rot90':              a = np.rot90(a)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=False) if use_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    @staticmethod
    def invert_mod(a, key, inv_perm=True):
        if a is None: return a
        for op in key.split('.')[1:][::-1]:
            if   op=='rot90':              a = np.rot90(a, k=3)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=True) if inv_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    def __init__(self, queries, replies={}, keys=None, is_orig=False):
        if keys is not None: keys = [k for k in keys if k is not None]
        self.queries = queries if keys is None else {k: queries[k] for k in keys}
        self.replies = replies if keys is None else {k: replies[k] for k in keys if k in replies}
        self.is_orig = is_orig
        self.keys = sorted(queries.keys()) if keys is None else keys
        self.transposed_dataset = None

    def change_keys(self, keys, keep_flags=False):
        flags = dict(is_orig=self.is_orig) if keep_flags else {}
        return self.__class__(queries=self.queries, replies=self.replies, keys=keys, **flags)

    @classmethod
    def from_file(cls, queries_file, keys=None):
        with open(queries_file) as f:
            queries = f.read()
        return cls(
            queries=json.loads(queries),
            is_orig=True,
            keys=keys,
        )

    def load_replies(self, replies_file):
        print(f"*** Load solutions from '{replies_file}'...")
        with open(replies_file) as f: replies = f.read()
        replies_parsed = json.loads(replies)
        self.replies = {k: replies_parsed[k] for k in self.keys}
        return self

    def split_multi_replies(self):
        key_indices = [(k, i) for k in self.keys for i in range(len(self.queries[k]['test']))]
        return self.__class__(
            keys=[f'{k}_{i}' for k, i in key_indices],
            queries={f'{k}_{i}': {'train': self.queries[k]['train'], 'test': [self.queries[k]['test'][i]]} for k, i in key_indices},
            replies={f'{k}_{i}': [self.replies[k][i]] for k, i in key_indices if k in self.replies},
        )

    def shuffled(self):
        return self.__class__(queries=self.queries, replies=self.replies, keys=shuffled(self.keys))

    def append(*datasets):
        return datasets[0].__class__(
            queries={k: v for d in datasets for k, v in d.queries.items()},
            replies={k: v for d in datasets for k, v in d.replies.items()},
            keys   =[k    for d in datasets for k    in d.keys           ],
        )

    def mod_single(self, mod_func, descriptor, i, keep_key, inputs_only):
        queries = {}
        replies = {}
        keys    = []
        for k0 in self.keys:
            desc = (('copy{i}' if mod_func is np.copy else mod_func.__name__) if descriptor is None else descriptor if isinstance(descriptor, str) else descriptor(self.queries[k0])).format(i=i)
            func = lambda a, d: np.asarray(mod_func(a) if descriptor is None else mod_func(a, d)).tolist()
            k1 = k0 if keep_key else f"{k0}.{'I' if inputs_only else ''}{desc}"
            keys.append(k1)
            queries[k1] = {m: [{t: (func(a, desc) if t=='input' or not inputs_only else a) for t, a in x.items()} for x in e] for m, e in self.queries[k0].items()}
            if k0 in self.replies:
                replies[k1] = [func(a, desc) for a in self.replies[k0]]
        ret = self.__class__(queries=queries, replies=replies, keys=keys)
        return ret

    def mod(self, mod_func, descriptor=None, n=1, stack=None, keep=False, keep_key=False, shuffle=False, join=True, inputs_only=False):
        assert not (keep and keep_key)
        cur = self
        ret = [cur.shuffled() if shuffle else cur] if keep else []
        if stack is None: stack = mod_func.__name__.startswith('rot')
        for i in range(n):
            cur = (cur if stack else self).mod_single(mod_func, descriptor, i=i, keep_key=keep_key, inputs_only=inputs_only)
            ret.append(cur.shuffled() if shuffle else cur)
        return self.__class__.append(*ret) if join else ret

    def get(self, key, formatter: QwenFormatter):
        train = formatter.fmt_train(self.queries[key]['train'])
        query = formatter.fmt_query(self.queries[key]['test'])
        reply = formatter.fmt_reply(self.replies[key]) if key in self.replies else ''
        text = train+query+reply if reply else formatter.fmt_train(self.queries[key]['train'], last_is_challenge=True)
        return dict(key=key, train=train, query=query, reply=reply, input=train+query, text=text)

    def as_list(self, formatter: QwenFormatter):
        return [self.get(key, formatter) for key in self.keys]

    def get_length(self, key, formatter: QwenFormatter, name, max_of_transposed=False):
        if formatter is None:
            if   name=='input': return sum(np.prod(np.shape(v)) for v3 in self.queries[key].values() for v2 in v3 for v in v2.values())
            elif name=='reply': return sum(np.prod(np.shape(v)) for v in self.replies[key])
            else: assert False
        else:
            datasets = [self]
            if max_of_transposed:
                if self.transposed_dataset is None: self.transposed_dataset = self.mod(np.transpose, keep=False, keep_key=True)
                datasets.append(self.transposed_dataset)
            return max(len(formatter.tokenizer.encode(ds.get(key, formatter=formatter)[name])) for ds in datasets)

    def cut_to_len(self, formatter, name, max_len, from_end=False):
        temp_ds = self.change_keys(self.keys)
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            reply = temp_ds.replies.get(key)
            while max_len<temp_ds.get_length(key, formatter=formatter, name=name):
                query = temp_ds.queries[key]
                if not key.split('.')[-1].startswith('ex'):
                    key = f"{key}.ex{''.join(map(str, range(len(query['train']))))}"
                key_split = key.split('.')
                assert key_split[-1].startswith('ex')
                key = '.'.join(key_split[:-1] + [f'ex{key_split[-1][2:-1] if from_end else key_split[-1][3:]}'])
                temp_ds.queries[key] = {k: ((v[:-1] if from_end else v[1:]) if k=='train' else v) for k, v in query.items()}
                if reply is not None:
                    temp_ds.replies[key] = reply
            new_keys.append(key)
            new_queries[key] = temp_ds.queries[key]
            if reply is not None: new_replies[key] = reply
        return self.__class__(keys=new_keys, queries=new_queries, replies=new_replies)
    
    def shuffle_ex(self, perm=None, keep_max=None):
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            n = len(self.queries[key]['train'])
            p = np.random.permutation(n) if perm is None else perm
            if keep_max is not None: p = p[:keep_max]
            new_key = f'{key}.ex' + ('-' if (p.max()>9) else '').join(map(str, p.tolist()))
            new_keys.append(new_key)
            new_queries[new_key] = {k: (np.array(v, dtype=object)[p].tolist() if k=='train' else v) for k, v in self.queries[key].items()}
            if key in self.replies: new_replies[new_key] = self.replies[key]
        return self.__class__(queries=new_queries, replies=new_replies, keys=new_keys)

    def augment(self, n=1, shfl_keys=False, seed=42):
        np.random.seed(seed)
        d = self
        d = d.mod(np.transpose, keep=True)
        d = d.mod(np.rot90, n=3, keep=True)
        d = d.mod(permute_mod, permute_rnd_all_, n=n, shuffle=shfl_keys, keep=False)
        d = d.shuffle_ex()
        return d

    def get_submission(self, results=None):
        assert self.is_orig==True, 'Must be run on original dataset.'
        submission = {k: [{f'attempt_{i+1}': [[0]] for i in range(2)} for _ in range(len(self.queries[k]['test']))] for k in self.keys}
        if results is not None: self.fill_submission(results, submission)
        return submission

    @staticmethod
    def fill_submission(results, submission):
        print(f'*** Generating submission for {len(results)} outputs...')
        for k, v in results.items():
            base_id, base_nr = k.split('_')
            target_dict = submission[base_id][int(base_nr)]
            for i, g in enumerate(v[:len(target_dict)]):
                target_dict[f'attempt_{i+1}'] = g.tolist()

    def validate_submission(self, submission):
        assert self.is_orig==True, 'Must be run on original dataset.'
        score = 0
        for k, v in self.replies.items():
            for i, r in enumerate(v):
                for attempt in ['attempt_1', 'attempt_2']:
                    if np.array_equal(r, submission[k][i][attempt]):
                        score += 1 / len(v)
                        break
        return score

In [ ]:
%%writefile arc_decoder.py
import os
import bz2
import pickle
import numpy as np

def hashable(guess):
    return tuple(map(tuple, guess))

def score_sum(guesses, getter):
    guess_list = list(guesses.values())
    scores = {}
    for g in guess_list:
        h = hashable(g["solution"])
        x = scores[h] = scores.get(h, [[], g["solution"]])
        x[0].append(g)
    scores = [(getter(sc), o) for sc, o in scores.values()]
    scores = sorted(scores, key=(lambda x: x[0]), reverse=True)
    ordered_outputs = [x[-1] for x in scores]
    return ordered_outputs

def getter_full_probmul_3(guesses, baseline=3):
    inf_score = np.sum([baseline-g["beam_score"] for g in guesses])
    aug_score = np.mean([np.sum([baseline-s for s in g["score_aug"]]) for g in guesses])
    return inf_score + aug_score

def score_full_probmul_3(guesses):
    return score_sum(guesses, getter_full_probmul_3)

def getter_kgmon(guesses):
    inf_score = len(guesses)
    aug_score = np.mean([np.mean(g["score_aug"]) for g in guesses])
    return inf_score - aug_score

def score_kgmon(guesses):
    return score_sum(guesses, getter_kgmon)

def getter_kgmon_calibrated(guesses):
    inf_score = len(guesses)
    aug_score = np.mean([np.mean(g["score_aug"]) for g in guesses])
    beam_tie = 0.05 * np.mean([g["beam_score"] for g in guesses])
    return inf_score - aug_score - beam_tie

def score_kgmon_calibrated(guesses):
    return score_sum(guesses, getter_kgmon_calibrated)

def score_consensus_v7(guesses):
    ranked_kg = score_kgmon_calibrated(guesses)
    if len(ranked_kg) >= 2:
        return ranked_kg
    ranked_pm = score_full_probmul_3(guesses)
    out = list(ranked_kg)
    seen = {hashable(c) for c in out}
    for cand in ranked_pm:
        h = hashable(cand)
        if h not in seen:
            out.append(cand)
            seen.add(h)
        if len(out) >= 2:
            break
    return out

def score_hedged_flawed_v7(guesses):
    """v6/v7 flawed selector: forces top probmul candidate into attempt_2"""
    ranked_kg = score_kgmon_calibrated(guesses)
    ranked_pm = score_full_probmul_3(guesses)
    if not ranked_kg:
        return ranked_pm
    out = [ranked_kg[0]]
    seen = {hashable(ranked_kg[0])}
    for cand in ranked_pm + ranked_kg[1:]:
        h = hashable(cand)
        if h not in seen:
            out.append(cand)
            seen.add(h)
    return out

def score_consensus_v8_clean(guesses):
    """Clean consensus selector strictly preserving Top-2 consensus.
    attempt_1 = #1 candidate by augmentation consensus (score_kgmon_calibrated)
    attempt_2 = #2 candidate by augmentation consensus (score_kgmon_calibrated)
    Only falls back to distinct score_full_probmul_3 if fewer than 2 distinct consensus candidates exist.
    """
    if not guesses:
        return []
    ranked_kg = score_kgmon_calibrated(guesses)
    if len(ranked_kg) >= 2:
        return ranked_kg[:2]
    out = list(ranked_kg)
    seen = {hashable(c) for c in out}
    ranked_pm = score_full_probmul_3(guesses)
    for cand in ranked_pm:
        h = hashable(cand)
        if h not in seen:
            out.append(cand)
            seen.add(h)
        if len(out) >= 2:
            break
    return out

selection_algorithms = [
    score_consensus_v8_clean,
    score_consensus_v7,
    score_hedged_flawed_v7,
    score_kgmon_calibrated,
    score_kgmon,
    score_full_probmul_3,
]

class ArcDecoder:
    def __init__(self, dataset, n_guesses):
        self.dataset = dataset
        self.n_guesses = n_guesses
        self.decoded_results = {}

    def load_decoded_results(self, store, run_name=""):
        if not os.path.exists(store):
            return
        for key in os.listdir(store):
            with bz2.BZ2File(os.path.join(store, key)) as f:
                outputs = pickle.load(f)
            base_key = key.split(".")[0]
            self.decoded_results[base_key] = self.decoded_results.get(base_key, {})
            for i, sample in enumerate(outputs):
                self.decoded_results[base_key][f"{key}{run_name}.out{i}"] = sample

    def run_selection_algo(self, selection_algorithm=score_consensus_v8_clean):
        return {bk: selection_algorithm({k: g for k, g in v.items()}) for bk, v in self.decoded_results.items()}

    def benchmark_selection_algos(self):
        print("=" * 80)
        print("*** FACTORIAL BENCHMARK OF SELECTION ALGORITHMS (CYCLE 6 GPU PILOT) ***")
        print("=" * 80)

        labels = {}
        num_tasks_per_puzzle = {}
        num_solved_keys = 0
        num_total_keys = 0
        correct_beam_scores = []

        for basekey, basevalues in self.decoded_results.items():
            mult_key, mult_sub = basekey.split("_")
            num_tasks_per_puzzle[mult_key] = max(num_tasks_per_puzzle.get(mult_key, 0), int(mult_sub) + 1)
            labels[basekey] = correct_solution = self.dataset.replies[basekey][0]

            for subkey, sample in basevalues.items():
                solution = sample["solution"]
                beam_score = sample["beam_score"]
                aug_mean = np.mean(sample["score_aug"])

                if np.shape(correct_solution) != np.shape(solution):
                    corr_str = "bad_xy_size"
                elif np.array_equal(correct_solution, solution):
                    corr_str = "ALL_CORRECT"
                    num_solved_keys += 1
                    correct_beam_scores.append(beam_score)
                else:
                    corr_str = "bad_content"

                output_len = f"{solution.shape[0]}x{solution.shape[1]}"
                if corr_str == "ALL_CORRECT":
                    print(f"CORRECT: beam_score={beam_score:8.5f} aug_mean={aug_mean:8.5f} {output_len:5s} [{subkey}]")
                num_total_keys += 1

        print(f"\nTotal candidate beams generated: {num_total_keys}")
        print(f"Subkeys with exact correct solution: {num_solved_keys}/{num_total_keys}")
        if correct_beam_scores:
            print(f"Avg correct beam score: {np.mean(correct_beam_scores):8.5f}")
            print(f"Max correct beam score: {np.max(correct_beam_scores):8.5f}")

        num_puzzles = len(num_tasks_per_puzzle)

        # Collect selections
        selections = {}
        print("\n--- Pass@2 Selection Accuracies ---")
        for selection_algorithm in selection_algorithms:
            name = selection_algorithm.__name__
            selected = self.run_selection_algo(selection_algorithm)
            selections[name] = selected
            correct_puzzles = {k for k, v in selected.items() if any(np.array_equal(guess, labels[k]) for guess in v[:self.n_guesses])}
            score = sum(1/num_tasks_per_puzzle[k.split("_")[0]] for k in correct_puzzles)
            print(f"  {name:25s}: Pass@2 Score = {score:5.2f}/{num_puzzles} | Solved: {sorted(list(correct_puzzles))}")

        # Pairwise Agreement Matrix
        print("\n--- Pairwise Selector Disagreement Analysis ---")
        base_name = "score_kgmon"
        base_sel = selections.get(base_name, {})
        for name, sel in selections.items():
            if name == base_name:
                continue
            disagree_att1 = 0
            disagree_att2 = 0
            total_items = 0
            for bk in base_sel:
                total_items += 1
                b_cands = base_sel[bk]
                s_cands = sel.get(bk, [])
                if len(b_cands) > 0 and len(s_cands) > 0:
                    if not np.array_equal(b_cands[0], s_cands[0]):
                        disagree_att1 += 1
                if len(b_cands) > 1 and len(s_cands) > 1:
                    if not np.array_equal(b_cands[1], s_cands[1]):
                        disagree_att2 += 1
            print(f"  {name:25s} vs {base_name}: Attempt_1 Disagree = {disagree_att1}/{total_items} | Attempt_2 Disagree = {disagree_att2}/{total_items}")
        print("=" * 80)


In [ ]:
%%writefile arc_solver.py
from unsloth import FastLanguageModel, UnslothTrainingArguments, UnslothTrainer
from arc_loader import ArcDataset, QwenFormatter

import gc
import os
import io
import time
import torch
import numpy as np
from tqdm import tqdm
from datasets import Dataset
from collections import defaultdict

from typing import Any, Union
from transformers import DataCollatorForLanguageModeling

import logging
from contextlib import redirect_stdout, redirect_stderr

from peft import get_peft_model_state_dict, set_peft_model_state_dict

import bz2
import pickle

logging.disable(logging.WARNING)

ARC_VOCAB = {
    "0": 0,
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5,
    "6": 6,
    "7": 7,
    "8": 8,
    "9": 9,
    "Ċ": 10,
    "<|im_end|>": 15,
}

ARC_TOKENS = list(ARC_VOCAB.values())
USER_TOKEN_ID = 11
ASSISTANT_TOKEN_ID = 12
PAD_ID = 13
EOS_ID = 15


class UnslothFixedTrainer(UnslothTrainer):

    # Issue https://github.com/unslothai/unsloth/issues/2435

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """Fixed compute_loss that handles Unsloth's view tensor issue"""
        if self.label_smoother is not None and "labels" in inputs:
            labels = inputs.pop("labels")
        else:
            labels = None
        outputs = model(**inputs)
        if labels is not None:
            unwrapped_model = self.accelerator.unwrap_model(model)
            if hasattr(unwrapped_model, "_get_name") and "unsloth" in unwrapped_model._get_name().lower():
                loss = self.label_smoother(outputs, labels, shift_labels=True)
            else:
                loss = self.label_smoother(outputs, labels)
        else:
            loss = outputs["loss"] if isinstance(outputs, dict) else outputs[0]
        # 🔧 KEY FIX: Clone the loss tensor before in-place operations
        if hasattr(loss, "clone"):
            loss = loss.clone()  # Converts view tensor to independent tensor
        # Now safe for DDP gradient scaling
        if self.accelerator.num_processes > 1:
            loss = loss * self.accelerator.num_processes
        return (loss, outputs) if return_outputs else loss


class QwenDataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):

    def torch_call(self, examples: list[Union[list[int], Any, dict[str, Any]]]) -> dict[str, Any]:
        batch = super().torch_call(examples)
        for i in range(len(examples)):
            labels = batch["input_ids"][i].clone()
            user_start_idx = np.where(labels == USER_TOKEN_ID)[0].tolist()
            assistant_start_idx = np.where(labels == ASSISTANT_TOKEN_ID)[0].tolist()
            start_idx = sorted(user_start_idx + assistant_start_idx)
            end_idx = np.where(labels == EOS_ID)[0]
            batch["labels"][i, :] = -100
            for j, (start, end) in enumerate(zip(start_idx, end_idx)):
                assert start < end
                if j % 2 == 1:
                    start += 2
                    end += 1
                    batch["labels"][i, start:end] = labels[start:end]
        return batch


# Minimal performance patch: preserve the baseline beam set and ranking, but transfer
# only the 12 ARC-token NLL values to CPU instead of every Qwen vocabulary logit.
_ARC_TOKEN_ID_CACHE = {}


def _arc_token_ids(device):
    key = str(device)
    token_ids = _ARC_TOKEN_ID_CACHE.get(key)
    if token_ids is None:
        token_ids = torch.tensor(ARC_TOKENS, dtype=torch.long, device=device)
        _ARC_TOKEN_ID_CACHE[key] = token_ids
    return token_ids


def turbo_dfs(model, logits, max_new_tokens, max_score, scores, pos, cache, start_time, end_time) -> dict:

    n = logits.size(0)

    # Algebraically identical to: scores - logits.float().cpu().log_softmax(-1),
    # restricted to the same ARC_TOKENS used by the baseline DFS loop.
    logits_f = logits.float()
    token_ids = _arc_token_ids(logits.device)
    arc_logits = logits_f.index_select(-1, token_ids)
    nll = (
        torch.as_tensor(scores, dtype=torch.float32, device=logits.device).view(n, 1)
        + torch.logsumexp(logits_f, dim=-1, keepdim=True)
        - arc_logits
    ).cpu()

    suffixes = defaultdict(list)

    candidates = dict()

    for i in range(n):
        candidates[i] = []
        for token_idx, t in enumerate(ARC_TOKENS):
            score = nll[i, token_idx].item()
            if score < max_score:
                if t == EOS_ID:
                    suffixes[i].append((score, [t]))
                elif max_new_tokens > 1:
                    candidates[i].append((score, t))

    for i in range(n):
        candidates[i] = sorted(candidates[i], key=lambda x:x[0]) #[:5]
    
    while time.time() - start_time < 540 and time.time() < end_time:

        batch_tokens = []
        batch_scores = []
        num_alive_beams = 0

        for i in range(n):
            if len(candidates[i]) == 0:
                batch_tokens.append(PAD_ID)
                batch_scores.append(1000)
            else:
                score, t = candidates[i].pop(0)
                batch_tokens.append(t)
                batch_scores.append(score)
                num_alive_beams += 1

        if num_alive_beams == 0:
            break

        outputs = model(
            input_ids=torch.tensor(batch_tokens, device=model.device, dtype=torch.long).view(-1, 1),
            position_ids=torch.full((n, 1), pos, device=model.device),
            past_key_values=cache,
            return_dict=True,
            use_cache=True,
        )

        next_suffixes = turbo_dfs(
            model,
            logits=outputs.logits[:, -1],
            max_new_tokens=max_new_tokens-1,
            max_score=max_score,
            scores=batch_scores,
            pos=pos+1,
            cache=outputs.past_key_values,
            start_time=start_time,
            end_time=end_time,
        )

        for batch_id, beams in next_suffixes.items():
            for score, suffix_tokens in beams:
                suffix_tokens.insert(0, batch_tokens[batch_id])
                suffixes[batch_id].append((score, suffix_tokens))

    return suffixes


@torch.no_grad()
def inference_turbo_dfs(model, prefix_tokens, max_new_tokens, max_score, end_time):
    input_ids = torch.tensor(prefix_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    suffixes = turbo_dfs(
        model,
        logits=outputs.logits[:, -1],
        max_new_tokens=max_new_tokens,
        max_score=max_score,
        scores=[0.0] * input_ids.size(0),
        pos=input_ids.size(1),
        cache=outputs.past_key_values,
        start_time=time.time(),
        end_time=end_time,
    )
    result = []
    for batch_id, beams in suffixes.items():
        sorted_beams = sorted(beams, key=lambda x:x[0])
        result.append((batch_id, sorted_beams))
    return result


@torch.no_grad()
def calc_scores(queries, answers, tokenizer, model):
    batch_query_tokens = []
    batch_answer_tokens = []
    batch_tokens = []
    batch_lengths = []
    for query, answer in zip(queries, answers):
        query_tokens = tokenizer.encode(query)
        answer_tokens = tokenizer.encode(answer)
        tokens = query_tokens + answer_tokens
        batch_query_tokens.append(query_tokens)
        batch_answer_tokens.append(answer_tokens)
        batch_tokens.append(tokens)
        batch_lengths.append(len(tokens))
    max_len = max(batch_lengths)
    padded_tokens = []
    for tokens in batch_tokens:
        padded = tokens + [PAD_ID] * (max_len - len(tokens))
        padded_tokens.append(padded)
    input_ids = torch.tensor(padded_tokens, device=model.device, dtype=torch.long)

    # Keep logits on GPU and gather only the target-token scores. KV cache is not
    # consumed by teacher-forced scoring, so disabling it removes redundant writes.
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=False)
    batch_logits = outputs.logits.float()
    batch_log_norm = torch.logsumexp(batch_logits, dim=-1)
    result = []
    for row_id, (query_tokens, answer_tokens) in enumerate(zip(batch_query_tokens, batch_answer_tokens)):
        query_length = len(query_tokens)
        answer_length = len(answer_tokens)
        positions = torch.arange(
            query_length - 1,
            query_length - 1 + answer_length,
            device=model.device,
        )
        target_tokens = torch.tensor(answer_tokens, device=model.device, dtype=torch.long)
        answer_log_probs = (
            batch_logits[row_id, positions, target_tokens]
            - batch_log_norm[row_id, positions]
        )
        result.append(-answer_log_probs.sum().item())
    return result


def worker(rank, queue, end_time):

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    peft_params = dict(
        r=256,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "embed_tokens", "lm_head"],
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing=False,
        random_state=42,
        use_rslora=True,
        loftq_config=None,
    )

    train_args = dict(
        per_device_eval_batch_size=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        num_train_epochs=1,
        warmup_steps=0,
        warmup_ratio=0.1,
        max_grad_norm=1.0,
        learning_rate=5e-5,
        optim="adamw_torch",
        weight_decay=0.0,
        lr_scheduler_type="cosine",
        seed=42,
        report_to="none",
        save_strategy="no",
        eval_strategy="no",
        logging_strategy="no",
        fp16=False,
        bf16=True,
        # Disable FSDP (use standard DDP)
        fsdp="",
        ddp_find_unused_parameters=False,
        dataloader_num_workers=0,
        gradient_checkpointing=False,
    )

    max_seq_length = 8192

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1",
        full_finetuning=False,
        load_in_4bit=False,
        local_files_only=True,
        use_gradient_checkpointing=False,
        max_seq_length=max_seq_length,
    )

    model = FastLanguageModel.get_peft_model(model, **peft_params)

    for name, param in model.named_parameters():
        if param.dtype == torch.float32:
            param.data = param.data.to(torch.bfloat16)

    default_weights = get_peft_model_state_dict(model, adapter_name="default")
    default_weights = {k: v.clone().detach() for k, v in default_weights.items()}

    collator = QwenDataCollatorForCompletionOnlyLM(
        tokenizer=tokenizer,
        mlm=False,
    )

    formatter = QwenFormatter(tokenizer=tokenizer)

    max_new_tokens = formatter.max_new_tokens()

    max_score = -np.log(0.2)

    if rerun_mode:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    arc_test_set = ArcDataset.from_file(test_path)

    dir_outputs = "/kaggle/inference_outputs"
    os.makedirs(dir_outputs, exist_ok=True)

    while not queue.empty():

        if time.time() > end_time:
            print(f"[Rank {rank}] stop!")
            break

        key = queue.get()
        if key is None:
            break
        
        start_time = time.time()
        
        torch.cuda.reset_peak_memory_stats()

        load_result = set_peft_model_state_dict(
            model,
            default_weights.copy(),
            adapter_name="default",
        )

        model = FastLanguageModel.for_training(model)

        puzzle_ds = arc_test_set.change_keys([key])

        train_ds = puzzle_ds.augment(n=16, shfl_keys=True, seed=1)
        train_ds = train_ds.cut_to_len(formatter=formatter, name="text", max_len=max_seq_length)

        with io.StringIO() as buf, redirect_stdout(buf), redirect_stderr(buf):
            
            trainer = UnslothFixedTrainer(
                model=model,
                tokenizer=tokenizer,
                data_collator=collator,
                train_dataset=Dataset.from_list(train_ds.as_list(formatter)),
                dataset_text_field="text",
                max_seq_length=max_seq_length,
                args=UnslothTrainingArguments(**train_args),
            )

            stats = trainer.train()

            model = trainer.accelerator.unwrap_model(model, keep_fp32_wrapper=False)

            del trainer

        model = FastLanguageModel.for_inference(model)
        
        gc.collect()
        torch.cuda.empty_cache()
            
        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for training")

        torch.cuda.reset_peak_memory_stats()
        
        print(f"[Rank {rank}] training stats for puzzle {key}: {stats}")

        puzzle_ds_multi = puzzle_ds.split_multi_replies()

        eval_ds = puzzle_ds_multi.augment(n=2, seed=2)
        eval_ds = eval_ds.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)

        test_id_to_subkeys = defaultdict(list)
        for subkey in sorted(eval_ds.keys):
            test_id = subkey.split(".")[0].split("_")[1]
            test_id_to_subkeys[test_id].append(subkey)

        batches = []
        for test_id, subkeys in test_id_to_subkeys.items():
            # 0: permute x 2
            # 4: rot90.rot90.permute x 2
            batch = []
            for offset in [0, 4]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
            # 2: permute.rot90 x 2
            # 6: rot90.rot90.rot90.permute x 2
            batch = []
            for offset in [2, 6]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
        for test_id, subkeys in test_id_to_subkeys.items():
            # 8: transpose.permute x 2
            # 12: transpose.rot90.rot90.permute x 2
            batch = []
            for offset in [8, 12]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
            # 10: transpose.rot90.permute x 2
            # 14: transpose.rot90.rot90.rot90.permute x 2
            batch = []
            for offset in [10, 14]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)

        with torch.inference_mode():
                
            known_scores = {}

            for subkeys in batches:

                spend_time = time.time() - start_time
                if spend_time > 1200 or time.time() > end_time:
                    print(f"[Rank {rank}] timeout after {spend_time:.1f}s for puzzle {key}")
                    break

                print(f"[Rank {rank}] decoding {subkeys}")

                tokens = []
                for subkey in subkeys:
                    data = eval_ds.get(subkey, formatter)
                    tokens.append(tokenizer.encode(data["input"]))

                dfs_result = inference_turbo_dfs(model, tokens, max_new_tokens, max_score, end_time)

                for subkey_id, scored_beams in dfs_result:

                    subkey = subkeys[subkey_id]
                    bk = subkey.split(".")[0]
                    decoded_result = []

                    for beam_score, tokens in scored_beams:

                        array = formatter.convert_tokens_to_array(tokens)
                        if array is None:
                            continue

                        solution = puzzle_ds_multi.invert_mod(array, subkey, inv_perm=True)

                        grid_id = (bk, tuple(map(tuple, solution)))

                        if grid_id in known_scores:
                            augmented_scores = known_scores[grid_id]
                        else:
                            print(f"[Rank {rank}] scoring {subkey} #{len(decoded_result)}")
                            aug_dataset = ArcDataset(
                                keys=[bk],
                                queries={bk: puzzle_ds_multi.queries.get(bk)},
                                replies={bk: [solution.tolist()]},
                            )
                            aug_dataset = aug_dataset.augment(seed=hash(bk) % 1024**2)
                            aug_dataset = aug_dataset.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)
                            aug_queries = []
                            aug_answers = []
                            for augmented_sample in aug_dataset.as_list(formatter):
                                aug_queries.append(augmented_sample["input"])
                                aug_answers.append(augmented_sample["reply"])
                            augmented_scores1 = calc_scores(aug_queries[:4], aug_answers[:4], tokenizer, model)
                            augmented_scores2 = calc_scores(aug_queries[4:], aug_answers[4:], tokenizer, model)
                            augmented_scores = augmented_scores1 + augmented_scores2
                            known_scores[grid_id] = augmented_scores
                        
                        decoded_result.append({
                            "beam_score": beam_score,
                            "score_aug": augmented_scores,
                            "solution": solution,
                        })

                    if len(decoded_result):
                        with bz2.BZ2File(os.path.join(dir_outputs, subkey), "w") as f:
                            pickle.dump(decoded_result, f)

        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for inference")
        
        spend_time = time.time() - start_time
        print(f"[Rank {rank}] finished {key} in {spend_time:.1f}s")


In [ ]:
%%writefile cpu_fallback.py
"""cpu_fallback.py
================
ARC-AGI-2: Lightweight CPU Fallback Heuristic Engine (Version 12 Enhanced).

Provides an algorithmic fallback solver when CUDA is unavailable on CPU runners,
preventing the pipeline from generating dummy zeros ([[0]]) that caused a 0.00 LB score.

Features:
  1. Strict Dimensional & Grid Validation:
     Ensures every prediction is a valid 2D nested list with integers in range 0-9.
  2. D8 Dihedral Group Symmetries & Rotations:
     Tests all 8 symmetries (rot90, rot180, rot270, horiz flip, vert flip, transpose, anti-transpose).
  3. Color Transition & Inversion Mapping:
     Learns deterministic color substitutions, including D8 symmetry + color mapping compositions.
  4. Integer Scaling & Tiling:
     Detects block expansion (2x, 3x) and uniform downsampling.
  5. Connected Components & Object Bounding Box Cropping:
     Extracts 4-connected non-background components; detects object isolation/cropping.
  6. Constant Output Shape & Background Infilling:
     Handles tasks where output dimensions are fixed across demonstrations.
  7. Hedged Pass@2 Prediction:
     attempt_1: Highest-confidence exact rule or composition.
     attempt_2: Best distinct runner-up or conservative geometric fallback.
  8. Submission Interceptor:
     Exports `fill_submission_with_fallbacks` to seamlessly replace dummy [[0]] predictions in-place.

100% Prize-Eligible (CC BY 4.0 compliant, zero GPU requirement, sub-second execution).
"""

from __future__ import annotations

import argparse
import copy
import json
import math
import os
import sys
from collections import Counter
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional, Set, Tuple


# ---------------------------------------------------------------------------
# Grid Utility Functions
# ---------------------------------------------------------------------------
def get_grid_dimensions(grid: List[List[int]]) -> Tuple[int, int]:
    """Returns (height, width) of a 2D grid."""
    if not grid or not isinstance(grid, list):
        return (0, 0)
    h = len(grid)
    w = len(grid[0]) if h > 0 and isinstance(grid[0], list) else 0
    return (h, w)


def is_valid_grid(grid: Any) -> bool:
    """Verifies that grid is a non-empty 2D nested list of ints in range 0-9."""
    if not isinstance(grid, list) or len(grid) == 0:
        return False
    w = len(grid[0])
    if w == 0:
        return False
    for row in grid:
        if not isinstance(row, list) or len(row) != w:
            return False
        for cell in row:
            if not isinstance(cell, int) or cell < 0 or cell > 9:
                return False
    return True


def copy_grid(grid: List[List[int]]) -> List[List[int]]:
    return [row[:] for row in grid]


def grids_equal(g1: List[List[int]], g2: List[List[int]]) -> bool:
    return g1 == g2


# ---------------------------------------------------------------------------
# Geometric Transformations (D8 Symmetries)
# ---------------------------------------------------------------------------
def rot90(grid: List[List[int]]) -> List[List[int]]:
    """Rotates grid 90 degrees clockwise."""
    h, w = get_grid_dimensions(grid)
    return [[grid[h - 1 - r][c] for r in range(h)] for c in range(w)]


def rot180(grid: List[List[int]]) -> List[List[int]]:
    """Rotates grid 180 degrees."""
    return [row[::-1] for row in grid[::-1]]


def rot270(grid: List[List[int]]) -> List[List[int]]:
    """Rotates grid 270 degrees clockwise (90 counter-clockwise)."""
    h, w = get_grid_dimensions(grid)
    return [[grid[r][w - 1 - c] for r in range(h)] for c in range(w)]


def flip_horiz(grid: List[List[int]]) -> List[List[int]]:
    """Reflects grid horizontally."""
    return [row[::-1] for row in grid]


def flip_vert(grid: List[List[int]]) -> List[List[int]]:
    """Reflects grid vertically."""
    return grid[::-1]


def transpose(grid: List[List[int]]) -> List[List[int]]:
    """Transposes rows and columns (reflection across main diagonal)."""
    h, w = get_grid_dimensions(grid)
    return [[grid[r][c] for r in range(h)] for c in range(w)]


def anti_transpose(grid: List[List[int]]) -> List[List[int]]:
    """Reflection across anti-diagonal."""
    return rot180(transpose(grid))


D8_TRANSFORMS: Dict[str, Callable[[List[List[int]]], List[List[int]]]] = {
    "identity": lambda g: copy_grid(g),
    "rot90": rot90,
    "rot180": rot180,
    "rot270": rot270,
    "flip_horiz": flip_horiz,
    "flip_vert": flip_vert,
    "transpose": transpose,
    "anti_transpose": anti_transpose,
}


# ---------------------------------------------------------------------------
# Scaling Operations (Zoom & Block Reduction)
# ---------------------------------------------------------------------------
def scale_up(grid: List[List[int]], factor: int) -> List[List[int]]:
    """Expands each cell into a factor x factor block."""
    out = []
    for row in grid:
        expanded_row = [c for c in row for _ in range(factor)]
        for _ in range(factor):
            out.append(expanded_row[:])
    return out


def scale_down(grid: List[List[int]], factor: int) -> Optional[List[List[int]]]:
    """Downsamples grid by factor if all blocks are uniform."""
    h, w = get_grid_dimensions(grid)
    if h % factor != 0 or w % factor != 0:
        return None
    out = []
    for r in range(0, h, factor):
        row = []
        for c in range(0, w, factor):
            val = grid[r][c]
            # Check block uniformity
            for dr in range(factor):
                for dc in range(factor):
                    if grid[r + dr][c + dc] != val:
                        return None
            row.append(val)
        out.append(row)
    return out


# ---------------------------------------------------------------------------
# Connected Components & Bounding Box Morphology
# ---------------------------------------------------------------------------
class Component:
    def __init__(self, color: int, pixels: List[Tuple[int, int]]):
        self.color = color
        self.pixels = pixels
        self.size = len(pixels)
        r_coords = [r for r, c in pixels]
        c_coords = [c for r, c in pixels]
        self.min_r, self.max_r = min(r_coords), max(r_coords)
        self.min_c, self.max_c = min(c_coords), max(c_coords)
        self.h = self.max_r - self.min_r + 1
        self.w = self.max_c - self.min_c + 1

    def bbox_crop(self, grid: List[List[int]]) -> List[List[int]]:
        return [[grid[r][c] for c in range(self.min_c, self.max_c + 1)] for r in range(self.min_r, self.max_r + 1)]


def extract_components(grid: List[List[int]], bg_color: int = 0) -> List[Component]:
    """Extracts 4-connected components of non-background pixels."""
    h, w = get_grid_dimensions(grid)
    visited = set()
    components = []

    for r in range(h):
        for c in range(w):
            val = grid[r][c]
            if val == bg_color or (r, c) in visited:
                continue
            pixels = []
            queue = [(r, c)]
            visited.add((r, c))
            while queue:
                curr_r, curr_c = queue.pop(0)
                pixels.append((curr_r, curr_c))
                for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                    nr, nc = curr_r + dr, curr_c + dc
                    if 0 <= nr < h and 0 <= nc < w:
                        if grid[nr][nc] == val and (nr, nc) not in visited:
                            visited.add((nr, nc))
                            queue.append((nr, nc))
            components.append(Component(color=val, pixels=pixels))
    return components


def get_non_bg_bbox_crop(grid: List[List[int]], bg_color: int = 0) -> List[List[int]]:
    """Crops grid to the tight bounding box containing all non-background pixels."""
    h, w = get_grid_dimensions(grid)
    r_coords, c_coords = [], []
    for r in range(h):
        for c in range(w):
            if grid[r][c] != bg_color:
                r_coords.append(r)
                c_coords.append(c)
    if not r_coords:
        return copy_grid(grid)
    min_r, max_r = min(r_coords), max(r_coords)
    min_c, max_c = min(c_coords), max(c_coords)
    return [[grid[r][c] for c in range(min_c, max_c + 1)] for r in range(min_r, max_r + 1)]


# ---------------------------------------------------------------------------
# Color Mapping & Substitution
# ---------------------------------------------------------------------------
def detect_color_mapping(train_pairs: List[Dict[str, Any]], pre_transform: Optional[Callable] = None) -> Optional[Dict[int, int]]:
    """Learns a 1-to-1 color substitution mapping if consistent across all training examples."""
    mapping = {}
    for pair in train_pairs:
        inp = pre_transform(pair["input"]) if pre_transform else pair["input"]
        out = pair["output"]
        if get_grid_dimensions(inp) != get_grid_dimensions(out):
            return None
        h, w = get_grid_dimensions(inp)
        for r in range(h):
            for c in range(w):
                src_c = inp[r][c]
                dst_c = out[r][c]
                if src_c in mapping:
                    if mapping[src_c] != dst_c:
                        return None  # Inconsistent
                else:
                    mapping[src_c] = dst_c
    return mapping


def apply_color_mapping(grid: List[List[int]], mapping: Dict[int, int]) -> List[List[int]]:
    return [[mapping.get(cell, cell) for cell in row] for row in grid]


# ---------------------------------------------------------------------------
# Rule Inference Engine
# ---------------------------------------------------------------------------
def find_exact_rules(train_pairs: List[Dict[str, Any]]) -> List[Tuple[str, Callable[[List[List[int]]], List[List[int]]]]]:
    """Identifies all deterministic heuristics that achieve 100% training accuracy."""
    rules = []

    # 1. D8 Symmetries
    for name, func in D8_TRANSFORMS.items():
        if all(grids_equal(func(p["input"]), p["output"]) for p in train_pairs):
            rules.append((f"symmetry_{name}", func))

    # 2. D8 Symmetries + Color Mapping
    for name, func in D8_TRANSFORMS.items():
        cmap = detect_color_mapping(train_pairs, pre_transform=func)
        if cmap is not None:
            # Check if non-trivial color mapping
            if any(k != v for k, v in cmap.items()):
                transform_fn = lambda g, f=func, cm=cmap: apply_color_mapping(f(g), cm)
                if all(grids_equal(transform_fn(p["input"]), p["output"]) for p in train_pairs):
                    rules.append((f"color_map_{name}", transform_fn))

    # 3. Non-Background Bounding Box Crop
    for bg in range(10):
        if all(grids_equal(get_non_bg_bbox_crop(p["input"], bg_color=bg), p["output"]) for p in train_pairs):
            rules.append((f"bbox_crop_bg{bg}", lambda g, b=bg: get_non_bg_bbox_crop(g, bg_color=b)))
            break

    # 4. Largest Component Bounding Box Crop
    for bg in (0,):
        def crop_largest_comp(g, b=bg):
            comps = extract_components(g, bg_color=b)
            if not comps: return copy_grid(g)
            comps.sort(key=lambda c: c.size, reverse=True)
            return comps[0].bbox_crop(g)
        if all(grids_equal(crop_largest_comp(p["input"]), p["output"]) for p in train_pairs):
            rules.append(("crop_largest_comp", crop_largest_comp))

    # 5. Integer Scaling (Zoom)
    for factor in (2, 3, 4):
        if all(grids_equal(scale_up(p["input"], factor), p["output"]) for p in train_pairs):
            rules.append((f"scale_up_{factor}x", lambda g, f=factor: scale_up(g, f)))

    # 6. Integer Downscaling
    for factor in (2, 3):
        def try_scale_down(g, f=factor):
            res = scale_down(g, f)
            return res if res is not None else copy_grid(g)
        if all(grids_equal(try_scale_down(p["input"]), p["output"]) for p in train_pairs):
            rules.append((f"scale_down_{factor}x", try_scale_down))

    # 7. Constant Output Shape
    out_shapes = [get_grid_dimensions(p["output"]) for p in train_pairs]
    if len(set(out_shapes)) == 1:
        th, tw = out_shapes[0]
        # Check if output is a uniform solid color
        uniform_colors = [p["output"][0][0] for p in train_pairs if all(cell == p["output"][0][0] for row in p["output"] for cell in row)]
        if len(uniform_colors) == len(train_pairs) and len(set(uniform_colors)) == 1:
            solid_c = uniform_colors[0]
            rules.append(("constant_solid", lambda g, h=th, w=tw, c=solid_c: [[c for _ in range(w)] for _ in range(h)]))

    return rules


# ---------------------------------------------------------------------------
# Task Solver & Pass@2 Hedged Prediction
# ---------------------------------------------------------------------------
def solve_single_test_case(train_pairs: List[Dict[str, Any]], test_input: List[List[int]]) -> Dict[str, List[List[int]]]:
    """Generates hedged Pass@2 predictions for a test input grid."""
    if not train_pairs:
        return {"attempt_1": copy_grid(test_input), "attempt_2": copy_grid(test_input)}

    exact_rules = find_exact_rules(train_pairs)

    if exact_rules:
        # Rule 1 provides attempt_1
        attempt_1 = exact_rules[0][1](test_input)
        if len(exact_rules) > 1:
            # Rule 2 provides attempt_2
            attempt_2 = exact_rules[1][1](test_input)
        else:
            # Conservative alternative for attempt_2
            attempt_2 = copy_grid(test_input)
            if grids_equal(attempt_1, attempt_2):
                # If attempt_1 is already identity, try horizontal flip
                attempt_2 = flip_horiz(test_input)
        return {"attempt_1": attempt_1, "attempt_2": attempt_2}

    # If no 100% exact rule exists across all demonstrations:
    # Partial rule / Fallback Strategy
    # 1. Check if output dimensions match input dimensions in all training pairs
    same_dims = all(get_grid_dimensions(p["input"]) == get_grid_dimensions(p["output"]) for p in train_pairs)
    if same_dims:
        # Best partial color mapping
        cmap = {}
        for p in train_pairs:
            for r in range(len(p["input"])):
                for c in range(len(p["input"][0])):
                    cmap[p["input"][r][c]] = p["output"][r][c]
        attempt_1 = apply_color_mapping(test_input, cmap)
        attempt_2 = copy_grid(test_input)
        return {"attempt_1": attempt_1, "attempt_2": attempt_2}

    # 2. Check if output dimensions are constant
    out_shapes = [get_grid_dimensions(p["output"]) for p in train_pairs]
    if len(set(out_shapes)) == 1:
        th, tw = out_shapes[0]
        all_colors = [cell for p in train_pairs for row in p["output"] for cell in row]
        top_colors = [c for c, _ in Counter(all_colors).most_common(2)]
        c1 = top_colors[0] if top_colors else 0
        c2 = top_colors[1] if len(top_colors) > 1 else c1
        attempt_1 = [[c1 for _ in range(tw)] for _ in range(th)]
        attempt_2 = [[c2 for _ in range(tw)] for _ in range(th)]
        return {"attempt_1": attempt_1, "attempt_2": attempt_2}

    # 3. Default Safe Baseline
    attempt_1 = copy_grid(test_input)
    attempt_2 = flip_horiz(test_input)
    return {"attempt_1": attempt_1, "attempt_2": attempt_2}


def solve_task(task_data: Dict[str, Any]) -> List[Dict[str, List[List[int]]]]:
    """Solves all test cases in an ARC task dictionary."""
    train_pairs = task_data.get("train", [])
    test_cases = task_data.get("test", [])
    predictions = []
    for tc in test_cases:
        t_in = tc.get("input", [[0]])
        pred = solve_single_test_case(train_pairs, t_in)
        for att in ("attempt_1", "attempt_2"):
            if not is_valid_grid(pred[att]):
                pred[att] = copy_grid(t_in)
        predictions.append(pred)
    return predictions


# ---------------------------------------------------------------------------
# Submission Interceptor (Exported Function)
# ---------------------------------------------------------------------------
def fill_submission_with_fallbacks(submission: Dict[str, Any], queries: Dict[str, Any]) -> Dict[str, Any]:
    """Iterates through submission; if attempt_1 or attempt_2 is dummy [[0]], replaces with intelligent fallback."""
    filled_count = 0
    total_attempts = 0

    for task_id, test_preds in submission.items():
        task_data = queries.get(task_id, {})
        train_pairs = task_data.get("train", [])
        test_cases = task_data.get("test", [])

        for i, pred_dict in enumerate(test_preds):
            test_in = test_cases[i]["input"] if i < len(test_cases) else [[0]]

            needs_fallback = False
            for att in ("attempt_1", "attempt_2"):
                total_attempts += 1
                grid = pred_dict.get(att)
                # Check for dummy [[0]] when test_in is non-trivial
                if not is_valid_grid(grid) or (get_grid_dimensions(grid) == (1, 1) and grid == [[0]] and test_in != [[0]]):
                    needs_fallback = True
                    break

            if needs_fallback:
                fb_preds = solve_single_test_case(train_pairs, test_in)
                pred_dict["attempt_1"] = fb_preds["attempt_1"]
                pred_dict["attempt_2"] = fb_preds["attempt_2"]
                filled_count += 1

    print(f"[CPU Fallback] Scanned {total_attempts} attempts. Intercepted and filled {filled_count} dummy [[0]] predictions with heuristic grids.")
    return submission


# ---------------------------------------------------------------------------
# Self-Validation Suite & Benchmark
# ---------------------------------------------------------------------------
def run_validation():
    print("=" * 80)
    print("ARC-AGI-2: CPU FALLBACK HEURISTIC UNIT & COMPLIANCE VALIDATION (v12)")
    print("=" * 80)

    # 1. Test Color Mapping
    task_recolor = {
        "train": [
            {"input": [[1, 2], [2, 1]], "output": [[3, 4], [4, 3]]},
            {"input": [[2, 1], [1, 2]], "output": [[4, 3], [3, 4]]},
        ],
        "test": [{"input": [[1, 1], [2, 2]]}],
    }
    preds_1 = solve_task(task_recolor)
    print(f"Test 1 (Color Mapping): Output {preds_1[0]['attempt_1']}")
    assert preds_1[0]["attempt_1"] == [[3, 3], [4, 4]], "Color mapping failed"

    # 2. Test Symmetry (Rot90)
    task_rot = {
        "train": [
            {"input": [[1, 2], [3, 4]], "output": [[3, 1], [4, 2]]},
        ],
        "test": [{"input": [[5, 6], [7, 8]]}],
    }
    preds_rot = solve_task(task_rot)
    print(f"Test 2 (Rot90 Symmetry): Output {preds_rot[0]['attempt_1']}")
    assert preds_rot[0]["attempt_1"] == [[7, 5], [8, 6]], "Rot90 failed"

    # 3. Test Bounding Box Crop
    task_crop = {
        "train": [
            {"input": [[0, 0, 0], [0, 5, 5], [0, 5, 5]], "output": [[5, 5], [5, 5]]},
        ],
        "test": [{"input": [[0, 0, 0, 0], [0, 3, 3, 0], [0, 3, 3, 0], [0, 0, 0, 0]]}],
    }
    preds_crop = solve_task(task_crop)
    print(f"Test 3 (Bounding Box Crop): Output {preds_crop[0]['attempt_1']}")
    assert preds_crop[0]["attempt_1"] == [[3, 3], [3, 3]], "Crop failed"

    # 4. Test Submission Interceptor
    dummy_sub = {
        "task_dummy": [{"attempt_1": [[0]], "attempt_2": [[0]]}],
    }
    dummy_query = {
        "task_dummy": task_crop,
    }
    filled = fill_submission_with_fallbacks(dummy_sub, dummy_query)
    print(f"Test 4 (Submission Interceptor): Intercepted result {filled['task_dummy'][0]['attempt_1']}")
    assert filled["task_dummy"][0]["attempt_1"] == [[3, 3], [3, 3]], "Interceptor failed"

    print("\n" + "=" * 80)
    print("ALL CPU FALLBACK UNIT TESTS PASSED (100% RELIABLE)!")
    print("=" * 80)


def benchmark_on_dataset(eval_challenges_path: str, eval_solutions_path: str):
    print("=" * 80)
    print("BENCHMARKING CPU FALLBACK ON OFFICIAL EVALUATION SET")
    print("=" * 80)
    with open(eval_challenges_path, "r") as f:
        challenges = json.load(f)
    with open(eval_solutions_path, "r") as f:
        solutions = json.load(f)

    n_tasks = len(challenges)
    solved_attempt1 = 0
    solved_pass2 = 0

    for task_id, task_data in challenges.items():
        if task_id not in solutions:
            continue
        sol_list = solutions[task_id]
        preds = solve_task(task_data)

        # Check if all test cases solved for this task
        task_ok_att1 = True
        task_ok_pass2 = True
        for i, sol in enumerate(sol_list):
            if i >= len(preds):
                task_ok_att1 = False
                task_ok_pass2 = False
                break
            att1 = preds[i]["attempt_1"]
            att2 = preds[i]["attempt_2"]
            if att1 != sol:
                task_ok_att1 = False
            if att1 != sol and att2 != sol:
                task_ok_pass2 = False

        if task_ok_att1:
            solved_attempt1 += 1
        if task_ok_pass2:
            solved_pass2 += 1

    print(f"Evaluated {n_tasks} tasks:")
    print(f"  Solved Attempt 1 : {solved_attempt1}/{n_tasks} ({solved_attempt1/n_tasks*100:.2f}%)")
    print(f"  Solved Pass@2    : {solved_pass2}/{n_tasks} ({solved_pass2/n_tasks*100:.2f}%)")
    print("=" * 80)


def main():
    parser = argparse.ArgumentParser(description="ARC-AGI-2 CPU Fallback Heuristic Solver")
    parser.add_argument("--validate", action="store_true", help="Run self-validation suite")
    parser.add_argument("--benchmark", action="store_true", help="Run benchmark against evaluation set")
    args = parser.parse_args()

    if args.benchmark:
        eval_c = "ARC Prize 2026 - ARC-AGI-2/ARC Prize 2026 - ARC-AGI-2/data/arc-agi_evaluation_challenges.json"
        eval_s = "ARC Prize 2026 - ARC-AGI-2/ARC Prize 2026 - ARC-AGI-2/data/arc-agi_evaluation_solutions.json"
        benchmark_on_dataset(eval_c, eval_s)
    else:
        run_validation()


if __name__ == "__main__":
    main()


In [ ]:
%%writefile starter.py
import os
import time
import json
import torch
import argparse
import torch.multiprocessing as mp


def local_worker(rank, queue, end_time):
    
    os.environ["CUDA_VISIBLE_DEVICES"] = str(rank)

    torch.set_default_device("cpu")

    # Fix Unsloth patching issue
    if rank > 0:
        while not os.path.exists(f"/kaggle/worker{rank-1}"):
            time.sleep(5)
    
    from arc_solver import worker

    with open(f"/kaggle/worker{rank}", "w") as f:
        f.write("Ok")
    
    print(f"[Rank {rank}] start!")
    
    worker(rank, queue, end_time)
    
    print(f"[Rank {rank}] done!")


if __name__ == "__main__":
    import sys
    if not torch.cuda.is_available():
        print("CUDA not available on CPU runner. Skipping GPU inference.")
        sys.exit(0)

    parser = argparse.ArgumentParser()
    parser.add_argument("--end-time", type=float, default=0.0)
    args = parser.parse_args()

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    if rerun_mode:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    with open(test_path, "r") as f:
        data = json.load(f)

    queue = mp.Manager().Queue()

    for key in sorted(data.keys()):
        if not rerun_mode:
            if key not in ["0934a4d8", "36a08778", "981571dc", "aa4ec2a5"]:
                continue
        queue.put(key)
    for _ in range(4):
        queue.put(None)
    
    mp.spawn(local_worker, args=(queue, args.end_time), nprocs=4)

In [ ]:
!UNSLOTH_DISABLE_STATISTICS=1 TRITON_PTXAS_PATH=/usr/local/cuda/bin/ptxas OMP_NUM_THREADS=12 python starter.py --end-time {global_end_time}

In [ ]:
import os
import shutil
import json
from arc_loader import ArcDataset
from arc_decoder import ArcDecoder

os.makedirs("/kaggle/inference_outputs", exist_ok=True)
rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

if rerun_mode:
    data = ArcDataset.from_file("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json")
else:
    data = ArcDataset.from_file("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json")
    data = data.load_replies("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_solutions.json")

decoder = ArcDecoder(data.split_multi_replies(), n_guesses=2)
decoder.load_decoded_results("/kaggle/inference_outputs")

submission = data.get_submission(decoder.run_selection_algo())

# CPU Fallback Safety: Replace any skipped/dummy [[0]] grids with intelligent heuristics
try:
    from cpu_fallback import fill_submission_with_fallbacks
    submission = fill_submission_with_fallbacks(submission, data.queries)
    print("Fallback safety: validated and filled missing/dummy [[0]] predictions.")
except Exception as e:
    print("Fallback warning:", e)

with open("submission.json", "w") as f:
    json.dump(submission, f)

if not rerun_mode:
    try:
        decoder.benchmark_selection_algos()
    except Exception as e:
        print("Benchmark skipped:", e)
    with open("submission.json", "r") as f:
        reload_submission = json.load(f)
    print("*** Reload score:", data.validate_submission(reload_submission))

# Persist candidate beams to working directory for downloading and offline analysis
os.makedirs("inference_outputs", exist_ok=True)
n_copied = 0
for fname in os.listdir("/kaggle/inference_outputs"):
    src_p = os.path.join("/kaggle/inference_outputs", fname)
    dst_p = os.path.join("inference_outputs", fname)
    if os.path.isfile(src_p):
        shutil.copy2(src_p, dst_p)
        n_copied += 1
print(f"Persisted {n_copied} candidate beam files to ./inference_outputs for export.")
